# Ajout de features pertinentes aux données

Ce notebook a pour but l'ajout de deux types de features pour nos données :
 - Des features externes en lien avec les biens immobiliers vendus
 - Des agrégats calculés à partir des données présentes

A la fin, on souhaite le jeu de données le plus propre possible pour l'utiliser directement pour l'entrainement de nos modèles. Le but est donc :
 - D'éviter la redondance de features 
 - De maximiser le nombre d'entrées complètes
 

## 1. Ajout de données communales

Dans cette partie on enrichi nos features avec des informations relatives à la commune du bien immobilier vendu.

Les données ajoutées sont :
- la distance à vol d'oiseau entre la voie du bien immobilier et le centre de la commune
- la population de la commune
- la densité de la commune 
- l'altitude moyenne de la commune

Pour ce faire, on utilise les informations fournies par le gouvernement sur toutes les communes et les adresses de France.

In [ ]:
import pandas as pd
import numpy as np
from haversine import haversine
import requests
import gzip
import io
import fastparquet as fp

from src.feature_engineering import (
    prepare_insee_code, fetch_ban_data, fetch_communes_data,
    enrich_with_geodata, add_postal_code_aggregates,
    add_temporal_aggregates, add_combined_features,
    clean_redundant_features
)

### 1.A. Charger les ventes DVF

In [ ]:
DATA_MODE = "dvf_merged_last5"  # Options: dvf_merged_last5 / dvf_merged_since2014
dvf_path = "./data/raw" + DATA_MODE + ".parquet"
df = pd.read_parquet(dvf_path)

# Reconstitution du code INSEE à partir du code commune et du code departement
df = prepare_insee_code(df)

### 1.B. Charger les fichiers BAN

In [ ]:
ban_df = fetch_ban_data()

Chargement https://adresse.data.gouv.fr/data/ban/adresses/latest/csv/adresses-01.csv.gz


Chargement https://adresse.data.gouv.fr/data/ban/adresses/latest/csv/adresses-02.csv.gz
Chargement https://adresse.data.gouv.fr/data/ban/adresses/latest/csv/adresses-03.csv.gz
Chargement https://adresse.data.gouv.fr/data/ban/adresses/latest/csv/adresses-04.csv.gz
Chargement https://adresse.data.gouv.fr/data/ban/adresses/latest/csv/adresses-05.csv.gz
Chargement https://adresse.data.gouv.fr/data/ban/adresses/latest/csv/adresses-06.csv.gz
Chargement https://adresse.data.gouv.fr/data/ban/adresses/latest/csv/adresses-07.csv.gz
Chargement https://adresse.data.gouv.fr/data/ban/adresses/latest/csv/adresses-08.csv.gz
Chargement https://adresse.data.gouv.fr/data/ban/adresses/latest/csv/adresses-09.csv.gz
Chargement https://adresse.data.gouv.fr/data/ban/adresses/latest/csv/adresses-10.csv.gz
Chargement https://adresse.data.gouv.fr/data/ban/adresses/latest/csv/adresses-11.csv.gz
Chargement https://adresse.data.gouv.fr/data/ban/adresses/latest/csv/adresses-12.csv.gz
Chargement https://adresse.data.

### 1.C. Charger les infos communes (population, densité et altitude moyenne)

In [ ]:
communes_url = "https://static.data.gouv.fr/resources/communes-et-villes-de-france-en-csv-excel-json-parquet-et-feather/20250221-162232/communes-france-2025.csv"
communes_df = fetch_communes_data()

### 1.D. Extrait voies uniques, coordonnées centre-ville, distance & Jointure finale

In [ ]:
df = enrich_with_geodata(df, ban_df, communes_df)

### 1.G. Sauvegarder le premier fichier enrichi

In [ ]:
df.to_parquet("./data/" + DATA_MODE + "_enriched1.parquet", index=False)
print("Fichier enrichi sauvegardé !")

Fichier enrichi sauvegardé !


Il manque des données pertinentes : évolutions démographiques géographique, des prix, du chomâge

## 2. Agrégat de features 

In [10]:
import pandas as pd

df = pd.read_parquet("./data/" + DATA_MODE + "_enriched1.parquet")
df["date_mutation"] = pd.to_datetime(df["date_mutation"])

## 2.A. Agrégats par code postal

In [11]:
grp_cp = df.groupby("code_postal")

agg_cp = grp_cp.agg(
    prix_m2_median_cp=("prix_m2", "median"),
    prix_m2_mean_cp=("prix_m2", "mean"),
    prix_m2_std_cp=("prix_m2", "std"),
    surface_m2_median_cp=("surface_m2", "median"),
    nb_pieces_mean_cp=("nb_pieces_principales", "mean"),
    transactions_cp=("prix_m2", "count")
).reset_index()

df = df.merge(agg_cp, on="code_postal", how="left")

## 2.B. Agrégats temporels

In [12]:
import pandas as pd
import numpy as np

df = df.sort_values(["code_postal", "date_mutation"]).reset_index(drop=True)

def fast_rolling_stats(group):
    dates = group["date_mutation"].astype("int64").to_numpy()
    prices = group["prix_m2"].to_numpy()

    n = len(group)
    medians = np.empty(n)
    counts = np.empty(n, dtype=np.int32)

    year_ns = np.int64(365*24*3600*1e9)

    for i in range(n):
        start = dates[i] - year_ns
        j = np.searchsorted(dates, start, side="left")

        window = prices[j:i+1]
        medians[i] = np.median(window)
        counts[i] = window.size

    return pd.DataFrame({
        "prix_m2_median_cp_last_12m": medians,
        "transactions_cp_last_12m": counts
    }, index=group.index)

df_stats = (
    df[["code_postal", "date_mutation", "prix_m2"]]
    .groupby("code_postal", group_keys=False)
    .apply(fast_rolling_stats)
)

df[["prix_m2_median_cp_last_12m", "transactions_cp_last_12m"]] = df_stats


/tmp/ipykernel_9906/3336580212.py:32: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(fast_rolling_stats)


## 2.C. Agrégats combinés

In [13]:
df["densite_x_population"] = df["densite_commune"] * df["population_commune"]
df["densite_population_ratio"] = df["population_commune"] / df["densite_commune"].replace(0, 1)
df["densite_altitude_ratio"] = df["densite_commune"] / df["altitude_moyenne_commune"].replace(0, 1)
df["attractivite_simple"] = df["population_commune"] / df["distance_centre_ville"].replace(0, 1)
df["distance_x_densite"] = df["distance_centre_ville"] * df["densite_commune"]

## 2.D. Sauvegarde

In [14]:
df.to_parquet("./data/" + DATA_MODE + "_enriched2.parquet", index=False)

print("Fichier enrichi sauvegardé sous " + DATA_MODE + "_enriched2.parquet")

Fichier enrichi sauvegardé sous dvf_merged_since2014_enriched2.parquet


## 3. Nettoyage des features redondantes


In [15]:
merged_enriched = "./data/" + DATA_MODE + "_enriched2.parquet"
df = fp.ParquetFile(merged_enriched).to_pandas()
print("Colonnes : ", df.columns)

Colonnes :  Index(['date_mutation', 'valeur_fonciere', 'type_local', 'surface_reelle_bati',
       'nb_pieces_principales', 'code_postal', 'commune', 'code_departement',
       'code_commune', 'code_voie', 'nombre_de_lots', 'surface_m2', 'prix_m2',
       'source_year', 'code_insee', 'lat', 'lon', 'distance_centre_ville',
       'densite_commune', 'altitude_moyenne_commune', 'population_commune',
       'prix_m2_median_cp', 'prix_m2_mean_cp', 'prix_m2_std_cp',
       'surface_m2_median_cp', 'nb_pieces_mean_cp', 'transactions_cp',
       'prix_m2_median_cp_last_12m', 'transactions_cp_last_12m',
       'densite_x_population', 'densite_population_ratio',
       'densite_altitude_ratio', 'attractivite_simple', 'distance_x_densite'],
      dtype='object')


In [16]:
cols_to_drop = ['commune','code_commune']   # code_insee déjà présent
df = df.drop(columns=cols_to_drop, errors='ignore')

Sauvegarde

In [17]:
df.to_parquet("./data/" + DATA_MODE + "_final.parquet", index=False)
print("Fichier enrichi sauvegardé !")

Fichier enrichi sauvegardé !


## 4. Statistiques

In [18]:
merged_final = "./data/" + DATA_MODE + "_final.parquet"
df = fp.ParquetFile(merged_final).to_pandas()

print("Colonnes : ", df.columns)

print("Nombre de lignes :", len(df))

nb_lignes_completes = df.dropna().shape[0]
print("Nombre de lignes complètes :", nb_lignes_completes)

print("\n", df.describe().drop(index=['25%', '50%', '75%'], errors='ignore'))

print("\n Un extrait : \n", df.head())

Colonnes :  Index(['date_mutation', 'valeur_fonciere', 'type_local', 'surface_reelle_bati',
       'nb_pieces_principales', 'code_postal', 'code_departement', 'code_voie',
       'nombre_de_lots', 'surface_m2', 'prix_m2', 'source_year', 'code_insee',
       'lat', 'lon', 'distance_centre_ville', 'densite_commune',
       'altitude_moyenne_commune', 'population_commune', 'prix_m2_median_cp',
       'prix_m2_mean_cp', 'prix_m2_std_cp', 'surface_m2_median_cp',
       'nb_pieces_mean_cp', 'transactions_cp', 'prix_m2_median_cp_last_12m',
       'transactions_cp_last_12m', 'densite_x_population',
       'densite_population_ratio', 'densite_altitude_ratio',
       'attractivite_simple', 'distance_x_densite'],
      dtype='object')
Nombre de lignes : 13393529
Nombre de lignes complètes : 10573141

                        date_mutation  valeur_fonciere  surface_reelle_bati  \
count                       13393529     1.339353e+07         1.339353e+07   
mean   2019-11-19 09:33:22.214144256     2